# 03 — Prompt Engineering

**Vai trò:** Pipeline Engineer · **Task:** S3-PE-06 (Yêu cầu 9.3)

Notebook này khám phá `PromptBuilder` (S3-PE-01) — bước **Build Prompt** trong luồng `query()`: ghép `system_prompt` + context chunks + câu hỏi thành một prompt hoàn chỉnh gửi cho LLM. Ta xác minh **Property 9** (prompt luôn chứa câu hỏi và toàn bộ context), rồi thực nghiệm đổi `system_prompt` qua `set_system_prompt()` để quan sát ảnh hưởng lên câu trả lời thật từ LLM cục bộ.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.generation.llm_client import OllamaClient
from src.generation.prompt_builder import PromptBuilder
from src.models import Chunk, ScoredChunk

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\lh222k\AI-Research-Assistant-with-RAG


## 1. `PromptBuilder` cơ bản — system prompt mặc định

`DEFAULT_SYSTEM_PROMPT` chỉ dẫn LLM trả lời **dựa trên context được cung cấp**. `build()` ghép system prompt + context (định dạng qua `format_context()`) + câu hỏi thành một chuỗi hoàn chỉnh (Yêu cầu 6.1, 6.2).

In [2]:
builder = PromptBuilder()
print("DEFAULT_SYSTEM_PROMPT:")
print(f"  {builder.system_prompt!r}\n")

contexts = [
    ScoredChunk(
        chunk=Chunk(chunk_id="c1", doc_id="doc_demo", content="RAG ket hop retrieval va generation de giam ao giac.", start_index=0, end_index=53),
        score=0.91, rank=1,
    ),
    ScoredChunk(
        chunk=Chunk(chunk_id="c2", doc_id="doc_demo", content="Indexing gom cac buoc: tai tai lieu, chia chunk, tao embedding, luu vector store.", start_index=53, end_index=134),
        score=0.78, rank=2,
    ),
]

question = "RAG hoat dong qua nhung buoc nao?"
prompt = builder.build(question, contexts)
print("--- Prompt hoan chinh ---")
print(prompt)

DEFAULT_SYSTEM_PROMPT:
  'Bạn là trợ lý nghiên cứu. Hãy trả lời câu hỏi DỰA TRÊN ngữ cảnh được cung cấp. Nếu không đủ thông tin, hãy nói rõ.'

--- Prompt hoan chinh ---
Bạn là trợ lý nghiên cứu. Hãy trả lời câu hỏi DỰA TRÊN ngữ cảnh được cung cấp. Nếu không đủ thông tin, hãy nói rõ.

### Ngữ cảnh
[Đoạn 1] (score=0.910, nguồn: doc_demo)
RAG ket hop retrieval va generation de giam ao giac.

[Đoạn 2] (score=0.780, nguồn: doc_demo)
Indexing gom cac buoc: tai tai lieu, chia chunk, tao embedding, luu vector store.

### Câu hỏi
RAG hoat dong qua nhung buoc nao?

### Trả lời


## 2. `format_context()` và trường hợp context rỗng (Yêu cầu 6.4)

Khi `contexts` rỗng (chưa index tài liệu nào, hoặc không tìm thấy đoạn liên quan), `PromptBuilder` vẫn phải tạo ra một prompt **hợp lệ** chứa system prompt và câu hỏi — không được để trống hay ném lỗi.

In [3]:
print("format_context(contexts khong rong):")
print(builder.format_context(contexts))

print("\nformat_context([]):")
print(builder.format_context([]))

empty_prompt = builder.build("Cau hoi khi chua co du lieu?", [])
assert builder.system_prompt in empty_prompt
assert "Cau hoi khi chua co du lieu?" in empty_prompt
print("\n--- Prompt voi context rong (van hop le) ---")
print(empty_prompt)

format_context(contexts khong rong):
[Đoạn 1] (score=0.910, nguồn: doc_demo)
RAG ket hop retrieval va generation de giam ao giac.

[Đoạn 2] (score=0.780, nguồn: doc_demo)
Indexing gom cac buoc: tai tai lieu, chia chunk, tao embedding, luu vector store.

format_context([]):
(Không có đoạn tài liệu nào liên quan được tìm thấy.)

--- Prompt voi context rong (van hop le) ---
Bạn là trợ lý nghiên cứu. Hãy trả lời câu hỏi DỰA TRÊN ngữ cảnh được cung cấp. Nếu không đủ thông tin, hãy nói rõ.

### Ngữ cảnh
(Không có đoạn tài liệu nào liên quan được tìm thấy.)

### Câu hỏi
Cau hoi khi chua co du lieu?

### Trả lời


## 3. Xác minh Property 9 — Prompt chứa đầy đủ câu hỏi và context

*Với mọi câu hỏi không rỗng và danh sách context chunks, prompt do `build()` tạo ra phải chứa nội dung câu hỏi và nội dung của tất cả context chunks* (design.md Phần 3, Property 9 — Validates Yêu cầu 6.1).

In [4]:
test_cases = [
    ("RAG hoat dong qua nhung buoc nao?", contexts),
    ("Cau hoi khong co context lien quan?", []),
    ("Chunk_size anh huong the nao den chat luong retrieval?", contexts[:1]),
]

for q, ctx in test_cases:
    p = builder.build(q, ctx)
    assert q in p, f"Property 9 vi pham: thieu cau hoi trong prompt cho {q!r}"
    for sc in ctx:
        assert sc.chunk.content in p, f"Property 9 vi pham: thieu noi dung context {sc.chunk.chunk_id!r}"
    print(f"OK  question={q!r}  (kem {len(ctx)} context)")

print("\nProperty 9 OK cho moi truong hop thu nghiem - prompt luon chua cau hoi + toan bo context.")

OK  question='RAG hoat dong qua nhung buoc nao?'  (kem 2 context)
OK  question='Cau hoi khong co context lien quan?'  (kem 0 context)
OK  question='Chunk_size anh huong the nao den chat luong retrieval?'  (kem 1 context)

Property 9 OK cho moi truong hop thu nghiem - prompt luon chua cau hoi + toan bo context.


## 4. Thực nghiệm prompt engineering — `set_system_prompt()`

`set_system_prompt()` thay đổi `system_prompt` dùng cho mọi lần `build()` tiếp theo (Yêu cầu 6.3) — đây chính là cách thực nghiệm các "phong cách" trả lời khác nhau. Nếu OLLAMA khả dụng, ta sinh câu trả lời thật cho cùng một câu hỏi + context với từng system prompt để so sánh trực tiếp.

In [5]:
SYSTEM_PROMPTS = {
    "default": PromptBuilder.DEFAULT_SYSTEM_PROMPT,
    "concise": "Ban la tro ly nghien cuu. Hay tra loi cau hoi DUA TREN ngu canh duoc cung cap, trong toi da 2 cau, khong giai thich dai dong.",
    "expert": "Ban la chuyen gia ky thuat ve he thong RAG. Hay tra loi chi tiet, dung thuat ngu chinh xac, va trich dan ro nguon (so doan) tu ngu canh duoc cung cap.",
}

client = OllamaClient(model_name="llama3", max_tokens=150)
available = client.is_available()
print(f"OLLAMA kha dung: {available}\n")

question_exp = "He thong RAG gom nhung buoc xu ly nao?"
for label, sys_prompt in SYSTEM_PROMPTS.items():
    builder.set_system_prompt(sys_prompt)
    p = builder.build(question_exp, contexts)
    assert builder.system_prompt == sys_prompt  # Yeu cau 6.3: build() dung system prompt moi nhat

    print(f"=== system_prompt = {label!r} ===")
    if available:
        answer = client.generate(p)
        print(answer[:350])
    else:
        print("(Bo qua sinh cau tra loi that - OLLAMA khong kha dung; chi minh hoa prompt khac nhau theo system_prompt.)")
    print()

builder.set_system_prompt(PromptBuilder.DEFAULT_SYSTEM_PROMPT)  # khoi phuc mac dinh
print(f"Da khoi phuc system_prompt mac dinh: {builder.system_prompt == PromptBuilder.DEFAULT_SYSTEM_PROMPT}")

OLLAMA kha dung: True

=== system_prompt = 'default' ===


Based on the provided context, I can answer your question as follows:

According to Đoạn 1 (RAG ket hop retrieval va generation de giam ao giac), it seems that the RAG system involves two main steps: retrieval and generation.

From Đoạn 2 (Indexing gom cac buoc: tai tai lieu, chia chunk, tao embedding, luu vector store), we can see that these two s

=== system_prompt = 'concise' ===


According to the provided context, the system RAG (Reinforced Attention-based Generative) consists of two main steps:

1. Retrieval: This step involves indexing and storing vectors in a vector store.
2. Generation: This step includes tasks such as tokenizing, chunking, and generating embeddings.

Therefore, the answer to the question "He thong RAG 

=== system_prompt = 'expert' ===


Based on the provided context and scoring information, I will provide a detailed answer using accurate technical terms and citing relevant sources (if available).

The Retrieval-Augmented Generation (RAG) system consists of several processing steps:

1. **Indexing**: This step involves creating an index that stores the input data, such as text or i

Da khoi phuc system_prompt mac dinh: True


## 5. Tổng kết

- `PromptBuilder.build()` luôn ghép **system prompt + context (định dạng qua `format_context`) + câu hỏi** thành một prompt hợp lệ — kể cả khi `contexts` rỗng (Yêu cầu 6.1, 6.2, 6.4).
- **Property 9** được xác minh trực tiếp: prompt luôn chứa nội dung câu hỏi và toàn bộ context chunks, với nhiều tổ hợp câu hỏi/context khác nhau.
- `set_system_prompt()` cho phép thực nghiệm nhiều "phong cách" trả lời (`concise`, `expert`, ...) mà không cần sửa code — chính là cơ chế PE dùng để tối ưu chất lượng câu trả lời từ LLM cục bộ (Yêu cầu 6.3).
- Đây là mảnh ghép cuối cùng trước bước **Generate** — kết hợp với `Retriever` (notebook [`02_retrieval_strategies.ipynb`](02_retrieval_strategies.ipynb)) và `OllamaClient.generate()`, ta có trọn vẹn `RAGPipeline.query()` như đã thấy ở [`01_rag_pipeline_basics.ipynb`](01_rag_pipeline_basics.ipynb).